# 📊 Bond Payment Data
<br>
<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 1px solid #eaeaea;">
  
  <a href="https://colab.research.google.com/github/PatrickJHess/Volume-Three-Chapter-Three/blob/master/colab/Colab_bond_payment_data.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 28px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Volume-Three-Chapter-Three/master?urlpath=lab/tree/notebooks/bond_payment_data.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">🚀</span> Launch Live in Binder
  </a>

  <a href="https://patrickjhess.github.io/Volume-Three-Chapter-Three/" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">⬅️</span> Return to Main Book
  </a>
</div>

📅 A bond's accrued interest is calculated based on scheduled payment amounts and dates, but its present value requires the actual amounts and dates. Although the payment amounts are fixed, the actual payment dates often deviate from the scheduled ones.

🗓️ Actual payment dates correspond to settlement dates, while scheduled payments are determined relative to the bond's maturity date. Since settlement days exclude weekends, holidays, and Good Friday, any payments scheduled for these days are advanced to the next available settlement day.
The `bond_payment_data` function in the notebook is used to determine both payment dates and amounts. This function relies on `adjust_bond_pay_dates`, which exemplifies the "Iceberg Principle." It utilizes the pandas_market_calendars library, a powerful tool providing valid trade dates for over fifty exchanges (including NYSE, LSE, and EUREX) and bond calendars for the U.S., U.K., and Japan.

🇺🇸 For this notebook, the U.S. calendar, `SIFMAUS`, is employed. The calendar is based on FED bank holidays and incorrectly treats Good Friday as a settlement date. To correct this the Pandas module, `tseries.holidays`and the rule `GoodFriday` are imported from the Pandas `tseries` module. This allows for a modification of the SIFMAUS-created calendar to accurately account for Good Friday, resulting in the correct set of settlement dates for U.S. bonds. The NumPy function `busday_offset` is used to account for non setlement dates, including Good Friday.

📓 The "Iceberg Principle" is further demonstrated in the companion notebook, *Bootstrapping Zero Prices*. That notebook leverages the `bond_payment_data`, `accrued_interest`, `FEDInvest`, and `clean_FEDInvest` functions to calculate zero prices from a sample of coupon bonds.


## Preparing the notebook

## 🛠️ Preparing the Notebook

<details>
<summary><b>👉 Click to Expand: 📦 Importing Libraries, Modules, and Functions</b></summary>

As a best practice, we always begin by importing our necessary dependencies in the very first code cell. Here the only library we import is the `fiancial_quant` package! ✨

**👀 Keep an eye out**: As we progress, pay attention to how the `financial_quant` package is imported as `fq`, and how every reference to its functions begins with fq.. 💡 This follows the exact same standard practice we demonstrated in Chapter One with NumPy (np) and Pandas (pd).

</details>

## 📦 Getting Functions from financial_quant package

<details>
<summary><b style="font-size:1.2em; color: #1976d2; cursor: pointer;">🔌 Professional Packaging: How GitHub Installations Work</b></summary>
<br>
<p><b>The Logic:</b><br>
Usually, Python looks for modules as <code>.py</code> files on your hard drive. Here, we are "tricking" Python into treating a string of text from a URL as a live library.</p>

<p><b>The Workflow:</b></p>
<ol>
<li><b>Fetch & Build:</b> The <code>%pip install git+https://...</code> command tells your Jupyter environment to clone the repository from GitHub and install the <code>financial_quant</code> package directly into your system's site-packages directory.</li>
<li><b>Import:</b> <code>import financial_quant as fq</code> loads the package into your notebook's memory and assigns it the quick alias <code>fq</code>.</li>
<li><b>Routing:</b> Behind the scenes, a special file called <code>__init__.py</code> acts as the package's "front door." It automatically gathers complex tools from deeply nested folders (like our fixed-income models and chart visualizers) and serves them up directly to the surface.</li>
<li><b>Execute:</b> You don't have to worry about where the files live. You just type <code>fq.one_y_axis() or fq.calc_ytm()</code>, and Python immediately knows where to route the request.</li>
</ol>

<p><b>Why do this?</b><br>
This is exactly how professional software engineering teams manage and distribute code. It keeps your notebooks incredibly clean, ensures everyone is using the exact same version of the math models, and guarantees your code is 100% portable to any cloud environment</p>
</details>

In [1]:
%pip install -q git+https://github.com/PatrickJHess/quant_repo.git 2> /dev/null
import financial_quant as fq

## 📓 This notebook explores two new features: the 📅 pandas_market_calendars library and the ↪️ busday_offset NumPy function. To help you experiment with these concepts, two code snippets are provided below.

🎄 Snippet 1: The first snippet uses the pandas_market_calendars library to find non-settlement days between December 21, 2025, and January 1, 2026. Christmas and New Year's are successfully detected, while Good Friday naturally does not occur within this date range.

⚙️ Snippet 2: The second snippet utilizes busday_offset to assign a valid settlement day to these holidays, as well as to regular weekends. This snippet assumes that the Datetime indexes dates, fed_holidays_idx, and Good_Friday_idx were previously created by the first snippet.

<details>
<summary><b>✂️ Click to see the snippet 1</b></summary>

**First Snippet Demonstrates `pandas_market_calendars`**
```python
import pandas as pd
from datetime import datetime, date
try:
  import pandas_market_calendars as mcal
except:
  %pip -q install pandas_market_calendars
  import pandas_market_calendars as mcal
from pandas.tseries.holiday import GoodFriday
# create datetime index
start=pd.Timestamp(2025,12,21)
end=pd.Timestamp(2026,1,10)
dates = pd.date_range(start=start, end=end)

# use the library to create the object
fed_cal = mcal.get_calendar('SIFMAUS')
fed_holidays = fed_cal.holidays().holidays

# get Good Fridays
Good_Fridays = GoodFriday.dates(start,end)

# create Pandas Datetime indexes
fed_holiday_idx = pd.DatetimeIndex(fed_holidays)
Good_Friday_idx = pd.DatetimeIndex(Good_Fridays)
# ceate filter for fed holidays to limit number
fed_holidays_start_end=(fed_holiday_idx>=start) & (fed_holiday_idx<=end)

# display the results for Panda Datetime indexes
display(fed_holiday_idx[fed_holidays_start_end])
display(Good_Friday_idx)
```
</details>

 🛑 run Snippet 1 before you run Snippet 2
<details>
<summary><b>✂️ Snippet 2 demonstrates NumPy busday_offset</b></summary>

```python
import numpy as np
# combine fed_and Good Friday Datetime indexes
combined_holidays_idx=fed_holiday_idx.union(Good_Friday_idx)

# Numpy dates must be datetime64
numpy_holidays =combined_holidays_idx.values.astype('datetime64[D]')

# Use NumPy for fully vectorized date math
actual_payment_dates = np.busday_offset(
    dates.values.astype('datetime64[D]'),
    offsets=0,
    roll='forward',
    holidays=numpy_holidays
)

# convert actual dates to datetime.date
settlement_dates = pd.to_datetime(actual_payment_dates).date

# dates that need adjusting
adjusted_dates=[{actual,settlement}
                         for actual,settlement in zip(dates.date,settlement_dates)
                         if actual!=settlement]
display(adjusted_dates)
```
</details>


<details>
<summary><b>🔍 Click to see adjust_bond_pay_dates</b></summary>

```python
def adjust_bond_pay_dates(dates,calendar='SIFMAUS'):
  """
  Adjusts bond payment dates to account for holidays and weekends.
  dates can be a scalar, pandas series, or numpy array (datetime.date,timestamp, or datetime64)
  """

  import pandas as pd
  import numpy as np
  import pandas_market_calendars as mcal
  from pandas.tseries.holiday import GoodFriday

  # Ensure dates is a DatetimeIndex
  if not pd.api.types.is_scalar(dates):
      dates = pd.DatetimeIndex(pd.to_datetime(dates))
  else:
      dates = pd.DatetimeIndex(pd.to_datetime([dates]))

  sifma = mcal.get_calendar(calendar)
  sifma_holidays = sifma.holidays().holidays
  good_fridays = GoodFriday.dates('2000-01-01', '2060-12-31')

  # Convert to DatetimeIndex and use .union() (which automatically deduplicates and sorts)
  sifma_idx = pd.DatetimeIndex(sifma_holidays)
  gf_idx = pd.DatetimeIndex(good_fridays)
  master_bond_holidays = sifma_idx.union(gf_idx)

  # Create a CustomBusinessDay offset using the combined holidays
  numpy_holidays = master_bond_holidays.values.astype('datetime64[D]')
```
</details>

## **Getting the actual payment dates and amounts**

🧾 Getting the actual payment dates and amounts
The `bond_pay_data` function is used to calculate both the exact payment dates and the corresponding payment amounts for a given bond.

* **📥 Inputs & Defaults**: The function requires the bond's maturity date and coupon rate. By default, the settlement date is set to the current day, and the payment frequency defaults to 2 (semi-annual payments). However, both of these can be manually specified if needed.

* **🛠️ Under the Hood**: To perform these calculations, bond_pay_data relies on two essential helper functions: scheduled_pay_dates (introduced in Chapter Two) and adjust_bond_pay_dates (built in this notebook).

* **📤 Outputs**: The function returns an array containing the calculated dates and payment amounts.

📈 These finalized dates and cash flows are exactly what we need for the next notebook, where they will be used to bootstrap zero prices from coupon bonds.

<details>
    
<summary> 🔍<b>Click to see the function</b></summary>

<br>

```python
def bond_pay_data(maturity, coupon, settlement=None, freq=2):
    '''
    Function calculates payment Dates And Amounts.
    maturity is a datetime object and coupon is a real number.
    Required arguments are maturity and annual coupon.
    If provided, the value of settlement is a datetime object;
    otherwise defaults to date.today()
    freq defaults to semi-annual but accepts freq equal
    to 1 for annual, 2 for semi-annal, 4 for quarterly, and 12 for monthly.
    The function assumes a par value of 100.
    Returns Numpy arrays of dates and amounts.

    Raises:
        TypeError: If maturity or settlement are not datetime objects.
        ValueError: If inputs are not logically valid (e.g., negative coupon,
                    maturity before settlement).
    '''
    from datetime import datetime, date
    from dateutil.relativedelta import relativedelta
    import pandas as pd
    import numpy as np
    from IPython.display import display, Markdown as md

    # Validate the data - maturity, coupon, settlement, freq
    def validate_date(datetime_object):
        # check for datetime or date
        if not isinstance(datetime_object, (datetime, date)):
            raise TypeError("Input must be a datetime or date object.")
        # convert datetime to date
        if isinstance(datetime_object, datetime):
            datetime_object = datetime_object.date()
        return datetime_object

    # maturity
    maturity = validate_date(maturity)

    # settlement
    if settlement is None:
        settlement = date.today()
    else:
        settlement = validate_date(settlement)

    # coupon
    try:
        coupon = float(coupon)
        if coupon < 0:
            raise ValueError("coupon rate cannot be negative.")
    except (ValueError, TypeError):
        raise ValueError("coupon must be a valid number.")

    # freq
    if int(freq) not in [1, 2, 4, 12]:
        display(md(f"### ⚠️ your assigned freq {freq} it must be (1, 2, 4, or 12)\n ### semi-annual assumed (2)."))
        freq = int(2)

    # check maturity greater than settlement
    if maturity <= settlement:
        raise ValueError("maturity must be greater than the settlement date")

    if coupon == 0:
        # Adjust maturity for non-settlement day and return date and face value
        adjust_maturity = adjust_bond_pay_dates(maturity)
        return np.array([adjust_maturity['Settlement'].dt.date]), np.array([100.0])

    # get scheduled payment dates from helper function scheduled_pay_dates
    scheduled_dates = scheduled_pay_dates(maturity, settlement, freq)

    # Pandas DataFrame Settlement desired column
    both_dates = adjust_bond_pay_dates(scheduled_dates)
    pay_dates=np.array(both_dates['Settlement'].dt.date)
    # calculate payments
    # coupon divided by freq at each date
    pay = np.full(len(pay_dates), coupon / freq)

    # Add principal payment as last cash payment
    pay[-1] += 100

    return pay_dates,pay
```
</details>

<div style="padding: 15px; border: 1px solid #e0e0e0; border-left: 5px solid #ffc107; background-color: #fffde7; border-radius: 4px; margin-bottom: 15px;">
  <b>✍️ Application:</b> Calculate the payment dates and amounts for two bonds that mature on August 31, 2035. One bond has a zero coupon and the other an annual coupon of 4.<br><br>
  <b>The Challenge:</b> Use the function <code>bond_pay_data</code>.<br>
  <b>Recall:</b> The 'Iceberg Principle' of functions that depend upon other functions that are imported.
</div>

<details>
<summary>🛟 <b>Need hints or a solution?</b></summary>

<br>

<details style="margin-left: 20px;">
<summary>💡 <b>Hints</b></summary>
<ul>
  <li><b>Imports:</b> Refer to the "Preparing notebook" section of this notebook</li>
  <li><b>Create the maturity and settlement dates:</b> <code>date(year, month, day)</code></li>
</ul>
</details>

<br>

<details style="margin-left: 20px;">
<summary>✅ <b>Example Of Solution</b></summary>

<br>

```python
# set the maturity and settlement date
maturity = date(2035, 8, 31)
settlement = date(2026, 4, 21)

# iterate through the coupons and display results
for coupon in [4, 0]:
    display(fq.bond_pay_data(maturity, coupon, settlement=settlement, freq=2))
```
</details>